# 쿼리 확장 — 찾는 가게 종류 키워드 생성 (Colab GPU, X8)

`docs/SERVICE_PIPELINE_PLAN.md` 11-16. 상황·추상 쿼리('보양식 재료', '학용품 파는 곳')에서 1단계가 놓치는 가게 종류를 LLM으로 풀어 쓴다
(Query2doc, Wang et al. EMNLP 2023; HyDE, Gao et al. ACL 2023). 서비스에서는 자주 나오는 쿼리를 미리 확장해 캐시하는 방식이 기본이다.

1. 로컬: `python scripts/export_x8_colab.py` → `colab_upload/x8/expand_queries.csv`를 Drive `내 드라이브/store-search-ai/query_expansion/`에 올림
2. Colab: 런타임 GPU(T4 가능) → 위에서부터 실행 → 같은 폴더에 `expansions_<모델>.csv`(query_id, query, expansion)
3. 로컬: 그 파일을 `results/service_pipeline/x8/`에 넣음

생성은 무작위 없이(greedy) — 다시 돌려도 같은 결과. 예시(few-shot)는 평가셋에 없는 쿼리로만 만들었다.


In [ ]:
import torch
assert torch.cuda.is_available(), "런타임 > 런타임 유형 변경 > GPU를 선택하세요"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install -U bitsandbytes

In [ ]:
from pathlib import Path

import pandas as pd

DIR = Path("/content/drive/MyDrive/store-search-ai/query_expansion")
MODEL = "Qwen/Qwen2.5-7B-Instruct"   # T4: 4bit로 올린다(약 5GB)
BATCH = 16
MAX_NEW_TOKENS = 48

In [ ]:
# @cell prompt
SYSTEM = ("너는 한국 전통시장·상점가 가맹점 검색을 돕는다. 사용자의 검색어를 보고, 그 사람이 찾아갈 가게의 종류와 그 가게가 파는 품목을 "
          "가게 취급품목에 실제로 쓰이는 짧은 한국어 명사로 3~6개 쉼표로 나열한다. 지명(동네·역·시장 이름)은 빼고, 설명이나 다른 말은 쓰지 않는다. "
          "가게 이름이나 브랜드면 그 가게의 업종을 쓴다.")
SHOTS = [("축의금 봉투", "문구, 봉투, 인쇄, 팬시"),
         ("고장난 시계", "시계, 시계수리, 금은방, 귀금속"),
         ("이사 박스", "포장자재, 박스, 생활용품, 철물"),
         ("망원시장 떡볶이", "분식, 떡볶이, 튀김, 순대")]


def messages(query: str) -> list[dict]:
    m = [{"role": "system", "content": SYSTEM}]
    for q, a in SHOTS:
        m += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return m + [{"role": "user", "content": query}]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))
model.eval()

In [ ]:
queries = pd.read_csv(DIR / "expand_queries.csv", encoding="utf-8-sig")
out_path = DIR / f"expansions_{MODEL.split('/')[-1]}.csv"
done = pd.read_csv(out_path, encoding="utf-8-sig") if out_path.exists() else pd.DataFrame(columns=["query_id", "query", "expansion"])
todo = queries[~queries["query_id"].isin(done["query_id"])]
print(f"확장할 쿼리 {len(todo)}개(이미 {len(done)}개)")
rows = []
for i in range(0, len(todo), BATCH):
    part = todo.iloc[i:i + BATCH]
    texts = [tok.apply_chat_template(messages(q), tokenize=False, add_generation_prompt=True) for q in part["query"]]
    enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.eos_token_id)
    outs = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    rows += [{"query_id": a, "query": b, "expansion": o.strip().split("\n")[0]} for a, b, o in zip(part["query_id"], part["query"], outs)]
    if (i // BATCH) % 10 == 0:
        pd.concat([done, pd.DataFrame(rows)]).to_csv(out_path, index=False, encoding="utf-8-sig", lineterminator="\n")
        print(i + len(part), "/", len(todo), rows[-1])
pd.concat([done, pd.DataFrame(rows)]).to_csv(out_path, index=False, encoding="utf-8-sig", lineterminator="\n")
print("[저장]", out_path)

In [ ]:
print(pd.read_csv(out_path, encoding="utf-8-sig").sample(20, random_state=0).to_string())